# 07 · ML · EDA e engenharia de features: valor do pedido

**Squad 2 · Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**
**Problema P3:** estimar o valor do pedido (ticket) a partir do contexto da compra, para antecipar receita e frete.

| | |
|---|---|
| Alvo | `valor_total` do pedido |
| Contexto previsto na planilha | hora, dia da semana, nº de itens, categoria dominante, método de pagamento, UF do endereço |
| Modelo (próxima etapa) | LightGBM Regressor, explicado por importância de variáveis / SHAP |
| Métricas | MAE e R², comparadas ao baseline "média do ticket por categoria dominante" |

**Fontes, só leitura.** Tempo real = Silvers da squad (`ecommerce_pedidos`, `ecommerce_itens_pedido`,
`ecommerce_enderecos`, `ecommerce_produtos`, `ecommerce_categorias`). Histórico = `raw/batch-data`, mesmo
layout, usado porque o tempo real tem poucas semanas de dados. Nenhuma tabela de origem é alterada.

**Saída:** uma linha por pedido, com as features escolhidas e a coluna `fonte` (`historico` ou `stream`), em
`squad2/grupo2/ml/features_valor_pedido` (Delta). O notebook 08 treina a partir dela.

In [ ]:
%run ./00_setup_config

## Parâmetros

`FATOR_PRECO_FORA` marca item com preço unitário pelo menos 5 vezes acima do preço de lista do catálogo
(achado da etapa 1: itens de R$ 1.000+ em produtos de R$ 10). A Silver de itens aprova esses itens porque a
regra T4 só barra preço baixo; aqui eles viram uma flag para a análise decidir o que fazer.

In [ ]:
from pyspark.sql import functions as F, Window as W
import matplotlib.pyplot as plt, pandas as pd

CAMINHO_SILVER_ENDERECOS, CAMINHO_SILVER_CATEGORIAS = f"{RAIZ_SQUAD}/silver/ecommerce_enderecos", f"{RAIZ_SQUAD}/silver/ecommerce_categorias"
CAMINHO_QUARENTENA_ENDERECOS = f"{RAIZ_SQUAD}/quarantine/ecommerce_enderecos"   # barrada por CEP/cadastro, mas UF sempre valida
CAMINHO_BATCH = f"abfss://raw@{_sufixo}/batch-data"
CAMINHO_FEATURES = f"{BASE}/ml/features_valor_pedido"
FATOR_PRECO_FORA, SEM_UF, FRACAO_TREINO = 5, "NAO_INFORMADO", 0.8
AZUL, AZUL_CLARO, CINZA = "#1F4E8C", "#5B8FC9", "#8C8C8C"
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.titleweight": "bold", "axes.edgecolor": "#595959", "axes.grid": True, "grid.alpha": .3})

## Leitura

O histórico não tem Silver: é o CSV do `batch-data`, lido com schema declarado (sem inferência) para os tipos
baterem com o tempo real; se o cabeçalho do arquivo mudar, a leitura falha em vez de deslocar colunas. Das
colunas de endereço só a UF é usada. As datas do histórico já vêm em horário local; as do tempo real são instantes e
passam para o fuso de São Paulo com `convert_timezone`, para que "hora" signifique a hora do cliente nas duas fontes.

In [ ]:
SCHEMAS_BATCH = {
    "ecommerce_pedidos": "id_pedido bigint, id_cliente bigint, id_endereco_entrega bigint, dt_pedido string, status_pedido string, valor_total double, valor_frete double, metodo_pagamento string, dt_previsao_entrega string, dt_ultima_atualizacao_status string",
    "ecommerce_itens_pedido": "id_item_pedido bigint, id_pedido bigint, sku string, quantidade bigint, preco_unitario double, desconto_aplicado double",
    "ecommerce_enderecos": "id_endereco bigint, id_cliente bigint, logradouro string, numero string, complemento string, bairro string, cep string, cidade string, estado string, latitude double, longitude double, apelido string, is_principal boolean",
    "ecommerce_produtos": "sku string, nome_produto string, descricao string, id_categoria bigint, preco_lista double, unidade_medida string, nome_marca string, is_ativo boolean",
    "ecommerce_categorias": "id_categoria bigint, nome_categoria string, id_categoria_pai double, nome_categoria_pai string, tipo_categoria string"}

def ler_csv(tabela):
    return spark.read.options(**adls_options).options(header=True, enforceSchema=False).schema(SCHEMAS_BATCH[tabela]).csv(f"{CAMINHO_BATCH}/{tabela}.csv")

def exigir_delta(caminho):
    df = ler_delta(caminho)
    if df is None: raise FileNotFoundError(f"Silver nao encontrada: {caminho.split('.net/')[-1]}")
    return df

FONTES = {
    "historico": dict(pe=ler_csv("ecommerce_pedidos"), it=ler_csv("ecommerce_itens_pedido"), en=ler_csv("ecommerce_enderecos"),
                      pr=ler_csv("ecommerce_produtos"), ca=ler_csv("ecommerce_categorias"), dt=F.to_timestamp_ntz("dt_pedido")),
    "stream": dict(pe=exigir_delta(CAMINHO_SILVER_PEDIDOS), it=exigir_delta(CAMINHO_SILVER),
                   en=exigir_delta(CAMINHO_SILVER_ENDERECOS).unionByName(exigir_delta(CAMINHO_QUARENTENA_ENDERECOS), allowMissingColumns=True),
                   pr=exigir_delta(CAMINHO_SILVER_PRODUTOS), ca=exigir_delta(CAMINHO_SILVER_CATEGORIAS),
                   dt=F.expr(f"convert_timezone('{FUSO_NEGOCIO}', dt_pedido)"))}
for nome, f in FONTES.items():
    print(f"{nome:9s} " + " | ".join(f"{k}={f[k].count():,}" for k in ("pe", "it", "en", "pr", "ca")))

## Uma linha por pedido

Os itens trazem produto e categoria (nível pai, 12 grupos como Pets, Bebidas, Limpeza). Por pedido:
nº de itens, quantidade total, nº de categorias, categoria dominante (a de maior quantidade no carrinho) e a
flag de item com preço fora da curva. A soma em reais dos itens é calculada **só para o teste de vazamento** da
próxima seção; ela não vai para o modelo.

Pedido sem item na Silver fica de fora (o ticket dele não tem carrinho para explicar). A UF sai em dois passos:

1. **Endereço do pedido.** No tempo real, a quarentena de endereços entra junto com a Silver: ela barra por CEP
   fora do padrão ou cadastro incompleto, mas a UF está sempre preenchida e válida. O join exige endereço **e**
   cliente iguais, porque ids pequenos de endereço do tempo real colidem com ids de endereços de outros clientes.
2. **Cadastro do cliente.** Quando o endereço do pedido não existe, a UF vem dos endereços que o cliente tem
   cadastrados (histórico + tempo real), só se todos estiverem na mesma UF. No histórico essa regra acerta a UF
   do endereço de entrega em 100% dos pedidos em que se aplica (98,8% se usasse a UF mais frequente de qualquer cliente).

Sem os dois, a UF vira `NAO_INFORMADO`. `uf_origem` registra de onde veio (`endereco`, `cliente`, `sem`).
Latitude e longitude não ajudam aqui: todo endereço já tem UF, e o pedido sem endereço também não tem coordenada
(o rastreamento não traz localização).

In [ ]:
def montar(fonte, pe, it, en, pr, ca, dt):
    cat = ca.select("id_categoria", F.coalesce("nome_categoria_pai", "nome_categoria").alias("categoria"))
    itc = (it.join(pr.dropDuplicates(["sku"]).select("sku", "id_categoria", "preco_lista"), "sku", "left").join(cat, "id_categoria", "left")
             .withColumn("valor_item", F.col("quantidade") * F.col("preco_unitario"))
             .withColumn("fora_curva", (F.col("preco_unitario") >= FATOR_PRECO_FORA * F.col("preco_lista")).cast("int")))
    por_cat = itc.groupBy("id_pedido", "categoria").agg(F.sum("quantidade").alias("q"), F.sum("valor_item").alias("v"))
    dom = por_cat.groupBy("id_pedido").agg(F.max_by("categoria", F.struct("q", "v")).alias("categoria_dominante"))
    agg = itc.groupBy("id_pedido").agg(F.count("*").alias("n_itens"), F.sum("quantidade").alias("qtd_total"), F.countDistinct("categoria").alias("n_categorias"),
            F.coalesce(F.max("fora_curva"), F.lit(0)).alias("flag_item_fora_curva"), F.sum("valor_item").alias("soma_itens_bruto"),
            F.sum(F.col("quantidade") * (F.col("preco_unitario") - F.col("desconto_aplicado"))).alias("soma_itens_liquido"))
    uf = en.select(F.col("id_endereco").alias("id_endereco_entrega"), "id_cliente", F.upper(F.trim("estado")).alias("uf")).dropDuplicates(["id_endereco_entrega"])
    metodo = F.regexp_replace(F.lower(F.trim("metodo_pagamento")), " de ", " ")   # "Cartão de Crédito" (histórico) = "cartão crédito" (stream)
    origem = F.when(F.col("uf").isNotNull(), "endereco").when(F.col("uf_cliente").isNotNull(), "cliente").otherwise("sem")
    return (pe.select("id_pedido", "id_endereco_entrega", "id_cliente", dt.alias("dt_local"), F.col("valor_total").cast("double"), metodo.alias("metodo_pagamento"))
              .join(agg, "id_pedido").join(dom, "id_pedido").join(uf, ["id_endereco_entrega", "id_cliente"], "left")
              .join(uf_cliente, "id_cliente", "left").drop("id_cliente")
              .withColumn("fonte", F.lit(fonte)).withColumn("uf_origem", origem).withColumn("uf_informada", (F.col("uf_origem") != "sem").cast("int"))
              .withColumn("uf", F.coalesce("uf", "uf_cliente", F.lit(SEM_UF))).drop("uf_cliente"))

# UF por cliente a partir de todos os endereços cadastrados; cliente com endereços em mais de uma UF fica de fora
_uf_cad = [en.select("id_cliente", F.upper(F.trim("estado")).alias("uf")) for en in (FONTES["historico"]["en"], FONTES["stream"]["en"])]
uf_cliente = (_uf_cad[0].unionByName(_uf_cad[1]).groupBy("id_cliente").agg(F.countDistinct("uf").alias("n_uf"), F.max("uf").alias("uf_cliente"))
                .filter("n_uf = 1").drop("n_uf"))

base = None
for nome, f in FONTES.items():
    df = montar(nome, **f)
    base = df if base is None else base.unionByName(df)
base = base.cache()
display(base.groupBy("fonte").agg(F.count("*").alias("pedidos"), F.min("dt_local").alias("de"), F.max("dt_local").alias("ate"),
        F.countDistinct(F.to_date("dt_local")).alias("dias"), F.round(F.avg("uf_informada") * 100, 1).alias("pct_com_uf")))
display(base.groupBy("fonte").pivot("uf_origem", ["endereco", "cliente", "sem"]).count().orderBy("fonte"))

## EDA 1 · Cobertura, nulos e duplicados

Quantos pedidos de cada fonte chegam até a base (têm itens) e quantos casam com endereço. Duplicado de
`id_pedido` na base indicaria que o join multiplicou linhas.

In [ ]:
for nome, f in FONTES.items():
    pe, it = f["pe"], f["it"]
    print(f"{nome:9s} pedidos={pe.count():,} | com itens={pe.join(it.select('id_pedido').distinct(), 'id_pedido', 'left_semi').count():,} | "
          f"id_pedido duplicado={pe.count() - pe.select('id_pedido').distinct().count()} | id_item duplicado={it.count() - it.select('id_item_pedido').distinct().count()}")
colunas = [c for c in base.columns if c not in ("fonte", "uf_origem")]
display(base.groupBy("fonte").agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas]))
print("id_pedido duplicado na base:", base.groupBy("fonte", "id_pedido").count().filter("count > 1").count())

## EDA 2 · Teste de vazamento

Se `valor_total` for a soma dos itens, qualquer variável em reais derivada do carrinho (soma, preço médio,
preço máximo) entrega a resposta ao modelo: o R² fica perto de 1 e não significa nada. O teste mede a
correlação e o erro médio entre o ticket e a soma dos itens.

In [ ]:
display(base.groupBy("fonte").agg(
    F.round(F.corr("valor_total", "soma_itens_bruto"), 4).alias("r_soma_bruta"), F.round(F.corr("valor_total", "soma_itens_liquido"), 4).alias("r_soma_liquida"),
    F.round(F.avg(F.abs(F.col("valor_total") - F.col("soma_itens_liquido"))), 2).alias("dif_media_R$")))

## EDA 3 · Distribuição do alvo

O ticket é assimétrico (cauda longa à direita), por isso o histograma está em escala log. As duas fontes
são comparadas lado a lado: se o perfil for muito diferente, um modelo treinado no histórico não serve
direto para pontuar o tempo real.

In [ ]:
quantis = base.groupBy("fonte").agg(F.count("*").alias("n"), F.round(F.avg("valor_total"), 2).alias("media"),
            *[F.round(F.percentile_approx("valor_total", p), 2).alias(f"p{int(p*100)}") for p in (.01, .25, .5, .75, .99)],
            F.round(F.max("valor_total"), 2).alias("max"), F.round(F.avg("n_itens"), 1).alias("itens_medio"), F.round(F.avg("qtd_total"), 1).alias("qtd_media"))
display(quantis)

hist = (base.withColumn("faixa", F.floor(F.log10("valor_total") * 10) / 10).groupBy("fonte", "faixa").count()
            .withColumn("pct", F.col("count") / F.sum("count").over(W.partitionBy("fonte")) * 100).toPandas())
fig, ax = plt.subplots(figsize=(11, 4))
for nome, cor in (("historico", AZUL), ("stream", AZUL_CLARO)):
    h = hist[hist.fonte == nome].sort_values("faixa")
    ax.bar(10 ** h.faixa, h.pct, width=10 ** (h.faixa + .1) - 10 ** h.faixa, align="edge", alpha=.7, color=cor, label=nome)
ax.set(xscale="log", xlabel="valor_total (R$, escala log)", ylabel="% dos pedidos", title="Distribuição do ticket por fonte"); ax.legend(); plt.show()

## EDA 4 · Itens com preço fora da curva

Quantos pedidos têm ao menos um item com preço ≥ 5× o de lista, e quanto da receita eles representam.

In [ ]:
display(base.groupBy("fonte").agg(F.sum("flag_item_fora_curva").alias("pedidos_flag"), F.round(F.avg("flag_item_fora_curva") * 100, 1).alias("pct_pedidos"),
        F.round(F.sum(F.col("valor_total") * F.col("flag_item_fora_curva")) / F.sum("valor_total") * 100, 1).alias("pct_receita"),
        F.round(F.avg(F.when(F.col("flag_item_fora_curva") == 1, F.col("valor_total"))), 2).alias("ticket_com_flag"),
        F.round(F.avg(F.when(F.col("flag_item_fora_curva") == 0, F.col("valor_total"))), 2).alias("ticket_sem_flag")))

## EDA 5 · Correlação das variáveis numéricas com o ticket

Pearson entre o ticket (e o log do ticket, que reduz o peso da cauda) e as variáveis numéricas candidatas.
Colunas derivadas entram aqui para serem avaliadas: `fim_de_semana` e `qtd_por_item` (quantidade média por linha do carrinho).

In [ ]:
base = (base.withColumn("hora", F.hour("dt_local")).withColumn("dia_semana", F.dayofweek("dt_local"))   # 1 = domingo
            .withColumn("fim_de_semana", F.col("dia_semana").isin(1, 7).cast("int"))
            .withColumn("periodo", F.when(F.col("hora") < 6, "madrugada").when(F.col("hora") < 12, "manha").when(F.col("hora") < 18, "tarde").otherwise("noite"))
            .withColumn("qtd_por_item", F.col("qtd_total") / F.col("n_itens")).withColumn("log_valor_total", F.log1p("valor_total")))
NUMERICAS = ["n_itens", "qtd_total", "n_categorias", "qtd_por_item", "hora", "dia_semana", "fim_de_semana", "flag_item_fora_curva", "uf_informada"]
ALVOS, corr = ("valor_total", "log_valor_total"), {}
for fonte in FONTES:   # com ANSI ligado, corr de coluna constante (ex.: uf_informada no historico) divide por zero: fica fora
    df = base.filter(F.col("fonte") == fonte)
    dp = df.agg(*[F.stddev(c).alias(c) for c in NUMERICAS]).first().asDict()
    variam = [c for c in NUMERICAS if dp[c]]
    r = df.agg(*[F.corr(c, a).alias(f"{c}|{a}") for c in variam for a in ALVOS]).first()
    for a in ALVOS: corr[(fonte, a)] = {c: (r[f"{c}|{a}"] if c in variam else None) for c in NUMERICAS}
corr = pd.DataFrame(corr).astype(float).round(3)
display(corr.reset_index().rename(columns={"index": "variavel"}).astype(str))

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(corr.shape[1]), [f"{f}\n{'log' if a.startswith('log') else 'R$'}" for f, a in corr.columns], fontsize=9); ax.set_yticks(range(len(corr)), corr.index); ax.grid(False)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]): ax.text(j, i, "-" if pd.isna(corr.values[i, j]) else f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlação com o ticket (Pearson)"); fig.colorbar(im, ax=ax, shrink=.8); plt.tight_layout(); plt.show()

## EDA 6 · Variáveis categóricas

Ticket médio por categoria dominante, método de pagamento, período do dia e UF. Uma categórica ajuda o
modelo quando o ticket médio muda bastante entre os grupos e cada grupo tem volume.

In [ ]:
def ticket_por(coluna, top=12):
    return (base.groupBy("fonte", coluna).agg(F.count("*").alias("pedidos"), F.round(F.avg("valor_total"), 2).alias("ticket_medio"))
                .withColumn("rk", F.row_number().over(W.partitionBy("fonte").orderBy(F.desc("pedidos")))).filter(f"rk <= {top}").drop("rk").toPandas())

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, coluna in zip(axes.flat, ("categoria_dominante", "metodo_pagamento", "periodo", "uf")):
    t = ticket_por(coluna).pivot(index=coluna, columns="fonte", values="ticket_medio").sort_values("historico")
    t.plot.barh(ax=ax, color=[AZUL, AZUL_CLARO], width=.8); ax.set(title=f"Ticket médio por {coluna}", xlabel="R$", ylabel="")
plt.tight_layout(); plt.show()
display(ticket_por("categoria_dominante").sort_values(["fonte", "pedidos"], ascending=[True, False]))

## Engenharia de features

**Escolha das colunas.** O que existia nas tabelas, o que entra no modelo e por quê:

| Coluna | Entra? | Motivo |
|---|---|---|
| `hora`, `dia_semana`, `periodo`, `fim_de_semana` | sim | contexto previsto no problema; `periodo` e `fim_de_semana` derivados de `dt_pedido` |
| `n_itens`, `qtd_total`, `n_categorias`, `qtd_por_item` | sim | tamanho e variedade do carrinho; não carregam preço |
| `categoria_dominante` | sim | perfil do carrinho; é também a chave do baseline |
| `metodo_pagamento` | sim | contexto previsto; categorias do histórico normalizadas para o formato do tempo real |
| `uf` + `uf_informada` | sim | contexto previsto; a flag separa "UF desconhecida" de uma UF real |
| `flag_item_fora_curva` | não como entrada | fica na tabela para filtrar ou analisar o erro do modelo; no momento da compra não é um contexto confiável |
| soma, preço médio ou máximo dos itens, `desconto_aplicado` | não | vazamento: `valor_total` é a soma dos itens (EDA 2) |
| `valor_frete`, `status_pedido`, `dt_previsao_entrega`, `dt_ultima_atualizacao_status` | não | só existem depois do pedido, ou dependem do próprio ticket |
| `uf_origem` | não como entrada | fica na tabela para auditar de onde veio a UF (`endereco`, `cliente`, `sem`) |
| `id_*`, `cidade`, `latitude`, `longitude`, `apelido` | não | identificadores; localização já resumida na UF |

**Categóricas em números.** `categoria_dominante`, `metodo_pagamento`, `periodo` e `uf` ganham um índice inteiro
(ordenado pela frequência no histórico; valor que só aparece no tempo real fica no fim). O LightGBM recebe o
índice como variável categórica, então não é preciso *one-hot*.

**Escala.** Não se normaliza: árvore de decisão só compara valores dentro da mesma coluna, então a diferença de
magnitude entre `qtd_total` e `hora` não dá mais peso a nenhuma delas. Se a etapa de comparação usar um modelo
linear, aí sim entra `StandardScaler`.

**Dimensionalidade.** São 12 entradas. Não há redução (PCA): seriam poucas colunas a menos, e componentes
principais apagariam a leitura de "qual variável pesa", que é o que o SHAP precisa mostrar.

**Alvo.** `valor_total` e `log_valor_total` ficam na tabela. O log reduz o efeito da cauda no treino; a métrica
volta para reais antes de comparar com o baseline.

**Treino e teste.** Divisão temporal dentro de cada fonte: os 80% mais antigos são treino, os 20% mais recentes,
teste. Embaralhar deixaria o modelo ver o "futuro".

In [ ]:
CATEGORICAS = ["categoria_dominante", "metodo_pagamento", "periodo", "uf"]
FEATURES = ["hora", "dia_semana", "fim_de_semana", "n_itens", "qtd_total", "n_categorias", "qtd_por_item", "uf_informada"] + [f"{c}_idx" for c in CATEGORICAS]

for c in CATEGORICAS:
    freq = base.groupBy(c).agg(F.sum((F.col("fonte") == "historico").cast("int")).alias("n_hist"), F.count("*").alias("n"))
    indice = freq.withColumn(f"{c}_idx", (F.row_number().over(W.orderBy(F.desc("n_hist"), F.desc("n"), c)) - 1).cast("int")).select(c, f"{c}_idx")
    base = base.join(F.broadcast(indice), c, "left")

ordem = F.percent_rank().over(W.partitionBy("fonte").orderBy("dt_local"))
features = (base.withColumn("conjunto", F.when(ordem < FRACAO_TREINO, "treino").otherwise("teste"))
                .select("fonte", "conjunto", "id_pedido", "dt_local", "uf_origem", *CATEGORICAS, *FEATURES, "flag_item_fora_curva", "valor_total", "log_valor_total"))
display(features.groupBy("fonte", "conjunto").agg(F.count("*").alias("pedidos"), F.min("dt_local").alias("de"), F.max("dt_local").alias("ate")).orderBy("fonte", "conjunto"))
for c in CATEGORICAS: display(features.groupBy(f"{c}_idx", c).pivot("fonte", list(FONTES)).count().orderBy(f"{c}_idx"))

## Gravação

Sobrescreve a tabela de features a cada execução (ela é derivada, não acumula histórico próprio) e relê para conferir.

In [ ]:
(features.write.format("delta").options(**adls_options).mode("overwrite").option("overwriteSchema", "true")
         .partitionBy("fonte").save(CAMINHO_FEATURES))
gravado = ler_delta(CAMINHO_FEATURES)
esperado = {r["fonte"]: r["count"] for r in features.groupBy("fonte").count().collect()}
obtido = {r["fonte"]: r["count"] for r in gravado.groupBy("fonte").count().collect()}
assert esperado == obtido, f"contagem divergente: {esperado} x {obtido}"
print(f"[ok] {CAMINHO_FEATURES.split('.net/')[-1]} | {obtido} | {len(FEATURES)} features")

## Resumo

Números da execução sobre os dados do lake em 16/11/2026.

| Achado | Histórico | Tempo real |
|---|---|---|
| Pedidos na base (com itens) | 90.335, jan/2024 a ago/2026 | 1.834 de 2.167, em 8 dias |
| Pedidos com UF | 100% | 99,6%: 1.303 pelo endereço (Silver + quarentena), 524 pelo cadastro do cliente, 7 sem |
| Correlação ticket × soma dos itens | 1,000 (diferença média R$ 0,00) | 0,998 |
| Ticket médio / mediana | R$ 842 / R$ 786 | R$ 274 / R$ 172 |
| Itens por pedido | 21,6 | 5,1 |
| Pedidos com item fora da curva | 0 | 88 (4,8%), com 22,1% da receita |

**Volume.** O tempo real sozinho é pouco para treinar. O treino principal usa o histórico;
o tempo real serve para testar e, depois, para ser pontuado.

**Vazamento.** `valor_total` é a soma dos itens. Nenhuma variável em reais do carrinho entra no modelo; se
entrasse, o R² ficaria perto de 1 sem prever nada.

**As fontes têm perfis diferentes.** No histórico o pedido é 3 vezes maior e tem 4 vezes mais itens. Um modelo
treinado só no histórico precisa ser medido no tempo real antes de pontuá-lo; essa comparação é o primeiro
teste do notebook 08.

**Valores que o histórico não tem.** No tempo real, `cartão débito` aparece em 511 pedidos (28%) e a categoria
Mercearia Salgada em 55; nenhum dos dois existe no histórico. Entre os pedidos com UF, a proporção SP/RJ/MG/ES é parecida com a do histórico; quase
todos os pedidos caem de manhã ou à tarde, o horário em que o gerador de dados fica ligado. Um modelo treinado
no histórico vai receber esses valores sem nunca tê-los visto.

**O que acompanha o ticket.** O tamanho do carrinho: `qtd_total` (r = 0,81 no histórico, 0,38 no tempo real),
`n_itens` (0,80 e 0,34) e `n_categorias` (0,65 e 0,18). A categoria dominante também separa: no histórico,
Matinais, Doces & Biscoitos tem ticket médio de R$ 1.081 e Adega & Destilados, de R$ 428. O contexto da
compra pesa pouco: hora, dia da semana e fim de semana têm |r| < 0,1 nas duas fontes, e no histórico o ticket
médio quase não muda entre métodos de pagamento ou UFs (R$ 830 a R$ 844).

**Itens fora da curva.** Só aparecem no tempo real e puxam o ticket desses pedidos para R$ 1.264 (contra R$ 224
nos demais). A flag fica na tabela para o 08 medir o erro com e sem eles.

**Saída.** 12 entradas por pedido, alvo em reais e em log, e a divisão temporal treino/teste, gravadas em
`squad2/grupo2/ml/features_valor_pedido`.